# 02 - Growing-season precipitation and dry / normal / wet years

Two quantities from the daily CoAgMet record at the Akron station:

- **P, the model feature:** precipitation accumulated from 1 October to 15 April. It is the same for every
  pixel in a year.
- **Year class:** the October-June total of each season is ranked against the 1993-2022 seasons.
  Below the 25th percentile is dry, above the 75th is wet, anything in between is normal.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
AWS_XLSX = Path("../data/raw/climate/Daily_AWS_data.xlsx")   # one sheet per year
OUT_DIR = Path("../data/processed")
TABLE_DIR = Path("../results/tables")

sheets = pd.read_excel(AWS_XLSX, sheet_name=None)
daily = pd.concat([s[["TIMESTAMP", "PrecipTotal"]] for name, s in sheets.items()
                   if str(name).strip().isdigit()], ignore_index=True)
daily["TIMESTAMP"] = pd.to_datetime(daily["TIMESTAMP"])
daily["PrecipTotal"] = pd.to_numeric(daily["PrecipTotal"], errors="coerce")
daily = daily.groupby(daily["TIMESTAMP"].dt.normalize())["PrecipTotal"].sum(min_count=1)
print(daily.index.min().date(), "to", daily.index.max().date())

In [ ]:
def season_total(year, end_month, end_day):
    start, end = pd.Timestamp(year - 1, 10, 1), pd.Timestamp(year, end_month, end_day)
    return daily.loc[start:end].sum()


YEARS = range(2019, 2025)
p_feature = pd.DataFrame({"Year": list(YEARS),
                          "P_Oct_to_Apr15_mm": [season_total(y, 4, 15) for y in YEARS]})
p_feature.to_csv(OUT_DIR / "precipitation_oct_apr15.csv", index=False)
p_feature.round(1)

## Year classes (Supplementary Table S4)

In [ ]:
seasons = pd.DataFrame({"Season": range(1993, 2025)})
seasons["P_Oct_to_Jun_mm"] = [season_total(y, 6, 30) for y in seasons["Season"]]

baseline = seasons[seasons["Season"].between(1993, 2022)]["P_Oct_to_Jun_mm"]
dry, wet = baseline.quantile(0.25), baseline.quantile(0.75)
seasons["Percentile"] = [(baseline <= p).mean() * 100 for p in seasons["P_Oct_to_Jun_mm"]]
seasons["Class"] = ["Dry" if p < dry else "Wet" if p > wet else "Normal" for p in seasons["P_Oct_to_Jun_mm"]]

print(f"dry below {dry:.1f} mm, wet above {wet:.1f} mm")
study = seasons[seasons["Season"] >= 2019]
study.to_csv(TABLE_DIR / "supp_table_s4_year_classes.csv", index=False)
study.round(1)